# Tranzia Decision Receipt Verification

This notebook demonstrates how to independently verify the integrity of a **Defensible Decision Receipt** issued by the Tranzia Platform.

### Why verify?
Trust is not claimed; it is proven. By hashing the decision data at the moment of generation, we ensure that no one—not even Tranzia—can alter the record of a decision without breaking the audit trail.

### Prerequisites
No special libraries required. Just standard Python.

In [ ]:
import json
import hashlib
import copy

## 1. Load the Receipt
We will load an example receipt. In a real scenario, this would be the JSON file you exported from the Tranzia Dashboard or API.

In [ ]:
# Load the example receipt included in the repo
with open('receipt_example_v1.json', 'r') as f:
    receipt = json.load(f)

print(f"Loaded Receipt ID: {receipt['receipt_id']}")
print(f"Timestamp: {receipt['created_at']}")

# Display the claimed integrity hash
claimed_hash = receipt['integrity']['canonical_receipt_hash']
print(f"\nCLAIMED HASH (from Receipt):\n{claimed_hash}")

## 2. Canonicalization
To reproduce the hash, we must serialize the JSON exactly as the server did. This process is called **canonicalization**.

**Rules:**
1. Sort keys alphabetically.
2. Remove insignificant whitespace (no spaces after commas or colons).

In [ ]:
def canonical_json(obj):
    """
    Produce stable JSON string for hashing.
    Matches Tranzia Backend: Sorted keys, compact separators (no spaces), UTF-8.
    """
    # Strict mode: Input must be native JSON types.
    return json.dumps(
        obj, 
        sort_keys=True, 
        separators=(',', ':'), 
        ensure_ascii=False
    )

## 3. Verify Integrity
We calculate the hash by:
1. Making a copy of the receipt.
2. Setting the `canonical_receipt_hash` field to an empty string `""` (since the hash cannot hash itself!).
3. Canonicalizing and hashing with SHA-256.

In [ ]:
def verify_receipt(original_receipt):
    # Work on a deep copy to avoid modifying the original data
    verifiable_receipt = copy.deepcopy(original_receipt)
    
    # 1. Zero out the hash field for calculation
    verifiable_receipt['integrity']['canonical_receipt_hash'] = ""
    
    # 2. Canonicalize
    canonical_str = canonical_json(verifiable_receipt)
    
    # 3. Compute SHA-256
    computed_hash = hashlib.sha256(canonical_str.encode('utf-8')).hexdigest()
    
    print(f"COMPUTED HASH:\n{computed_hash}")
    
    # 4. Compare
    expected = original_receipt['integrity']['canonical_receipt_hash']
    if computed_hash == expected:
        print("\n✅ SUCCESS: The receipt is Authentic and Unmodified.")
        return True
    else:
        print("\n❌ FAILURE: The receipt has been TAMPERED with!")
        return False

# Run verification on the loaded receipt
verify_receipt(receipt);

## 4. Detect Tampering (Demo)
Let's see what happens if someone tries to maliciously alter the safety score.

In [ ]:
# Simulate a malicious edit
tampered_receipt = copy.deepcopy(receipt)

# Hacker changes the score from 8.5 to 10.0
tampered_receipt['baseline_assessment']['risk_score_0_10'] = 10.0
print("⚠️ Tampering detected: Changed risk score to 10.0")

# Attempt to verify the tampered receipt
verify_receipt(tampered_receipt);